In [1]:
from tdc.single_pred import Tox
from rdkit import Chem, RDLogger
from rdkit.Chem import Lipinski, Descriptors, Crippen,rdMolDescriptors, GraphDescriptors, QED
from rdkit.Chem import Fragments, AllChem
from rdkit.Chem.rdMolDescriptors import (
    CalcTPSA, CalcKappa1, CalcKappa2, CalcKappa3,
    CalcChi0v, CalcChi1v, CalcChi2v, CalcChi3v, CalcChi4v,
    CalcChi0n, CalcChi1n, CalcChi2n, CalcChi3n, CalcChi4n,
)

RDLogger.DisableLog('rdApp.*')

from copy import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

# Local utilities
import sys
sys.path.insert(0, '.')

random_state = 0

In [2]:
data = Tox(name = 'LD50_Zhu', path= '../data/LD50_Zhu')
split = data.get_split()

Found local copy...
Loading...
Done!


In [3]:
split['train']['mol'] = split['train'].Drug.apply(lambda x: Chem.MolFromSmiles(x))
split['train']['canonical_smiles'] = split['train']['mol'].apply(
    lambda x: Chem.MolToSmiles(x) if x is not None else None
)

In [4]:
df_train = split['train']

In [5]:
def safe_desc(func, mol):
    try:
        return func(mol) if mol is not None else None
    except:
        return None

from rdkit.Chem import Descriptors

descriptors = {
    # massa
    'ExactMolWt': Descriptors.ExactMolWt,
    'MolWt': Descriptors.MolWt,
    'HeavyAtomMolWt': Descriptors.HeavyAtomMolWt,

    # cariche
    'MaxAbsPartialCharge': Descriptors.MaxAbsPartialCharge,
    'MaxPartialCharge': Descriptors.MaxPartialCharge,
    'MinAbsPartialCharge': Descriptors.MinAbsPartialCharge,
    'MinPartialCharge': Descriptors.MinPartialCharge,

    # elettroni
    'NumValenceElectrons': Descriptors.NumValenceElectrons,

    # fingerprint density
    'FpDensityMorgan1': Descriptors.FpDensityMorgan1,
    'FpDensityMorgan2': Descriptors.FpDensityMorgan2,
    'FpDensityMorgan3': Descriptors.FpDensityMorgan3,
}

for name, func in descriptors.items():
    df_train[name] = df_train['mol'].apply(lambda m: safe_desc(func, m))

df_train['fp'] = df_train['mol'].apply(
    lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=1024)
)

## Export Final Dataset

In [6]:
# Export dataset (exclude non-serializable columns: mol, fp)
export_cols = ['Drug_ID', 'Y', 'canonical_smiles'] + list(descriptors.keys())

df_export = df_train[export_cols].copy()

# Create output directory if needed
import os
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

# Export to CSV
csv_path = f'{output_dir}/toxicity_dataset_with_descriptors.csv'
df_export.to_csv(csv_path, index=False)
print(f"✓ CSV exported: {csv_path} (shape: {df_export.shape})")

# Also export as pickle for reproducibility (includes mol objects)
pkl_path = f'{output_dir}/toxicity_dataset_full.pkl'
df_train.to_pickle(pkl_path)
print(f"✓ Full dataset (with RDKit objects) exported: {pkl_path}")

✓ CSV exported: ../data/processed/toxicity_dataset_with_descriptors.csv (shape: (5170, 14))
✓ Full dataset (with RDKit objects) exported: ../data/processed/toxicity_dataset_full.pkl
